# KhazinaSmart — Hyperparameter Tuning

Uses Optuna to find optimal XGBoost hyperparameters with TimeSeriesSplit cross-validation.

In [1]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from src.feature_engineering import get_train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/features_final.csv', parse_dates=['Date'])
X_train, X_test, y_train, y_test = get_train_test_split(df)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

[Split] Train: 24,400 rows | Test: 3,000 rows
Train: (24400, 25) | Test: (3000, 25)


## Optuna Hyperparameter Optimization

In [2]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    USE_OPTUNA = True
except ImportError:
    USE_OPTUNA = False
    print('Optuna not installed — using GridSearch fallback')

if USE_OPTUNA:
    tscv = TimeSeriesSplit(n_splits=3)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
        }
        scores = []
        for train_idx, val_idx in tscv.split(X_train):
            Xt, Xv = X_train.iloc[train_idx], X_train.iloc[val_idx]
            yt, yv = y_train.iloc[train_idx], y_train.iloc[val_idx]
            model = XGBRegressor(**params)
            model.fit(Xt, yt, verbose=False)
            preds = model.predict(Xv)
            scores.append(np.sqrt(mean_squared_error(yv, preds)))
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    print(f'Best RMSE: {study.best_value:,.0f}')
    print(f'Best params: {study.best_params}')

  0%|          | 0/30 [00:00<?, ?it/s]

Best RMSE: 2,900
Best params: {'n_estimators': 909, 'max_depth': 5, 'learning_rate': 0.03548156923979383, 'subsample': 0.9648513548238948, 'colsample_bytree': 0.8538380801136655, 'min_child_weight': 4}


## Train Tuned Model

In [3]:
if USE_OPTUNA:
    best_params = study.best_params
else:
    best_params = {
        'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 3,
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
    }

tuned_model = XGBRegressor(**best_params)
tuned_model.fit(X_train, y_train, verbose=False)
tuned_preds = tuned_model.predict(X_test)

from sklearn.metrics import mean_absolute_error, r2_score
rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))
mae = mean_absolute_error(y_test, tuned_preds)
mape = np.mean(np.abs((y_test - tuned_preds) / (y_test + 1))) * 100
r2 = r2_score(y_test, tuned_preds)
print(f'Tuned XGBoost — RMSE: {rmse:,.0f} | MAE: {mae:,.0f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')

Tuned XGBoost — RMSE: 2,655 | MAE: 1,824 | MAPE: 7.70% | R2: 0.9695


## Optimization History

In [4]:
if USE_OPTUNA:
    import optuna.visualization as vis
    fig = vis.plot_optimization_history(study)
    fig.show()

## Save Tuned Model

In [5]:
os.makedirs('../models', exist_ok=True)
joblib.dump(tuned_model, '../models/xgb_tuned.pkl')
print(f'Tuned model saved to models/xgb_tuned.pkl')
print(f'Final RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')

Tuned model saved to models/xgb_tuned.pkl
Final RMSE: 2,655 | MAPE: 7.70% | R2: 0.9695
